In [1]:
import json
import numpy as np
import pandas as pd
from io import StringIO
import textwrap
from model_inference.gpt import *
from utils.table_utils import *

# Table parsing test

In [2]:
path = '../data/livesum/test.json'

In [3]:
df = pd.read_json(path)

In [4]:
print(df.head())

                                                text  \
0  And we're off for the first half. Player27(Awa...   
1  The game is underway with the start of the fir...   
2  The game is underway with the start of the fir...   
3  And we're off for the first half. Player26(Awa...   
4  The game is underway with the start of the fir...   

                                               table        id  
0  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25513332  
1  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25513360  
2  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25600389  
3  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25617902  
4  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25892175  


In [5]:
print(df.columns)

Index(['text', 'table', 'id'], dtype='object')


In [6]:
idx = 2

In [7]:
print(df['table'][idx])

Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,Corner Kicks,Free Kicks,Offsides<NEWLINE>Away Team,2,10,16,5,1,7,7,1<NEWLINE>Home Team,0,19,7,1,0,8,15,1


In [8]:
table_string = df['table'][idx]
table_string = table_string.replace('<NEWLINE>', '\n')

In [9]:
table_string_io = StringIO(table_string)

In [10]:
df_table = pd.read_csv(table_string_io)

In [11]:
print(df_table.to_string(index=False))

     Team  Goals  Shots  Fouls  Yellow Cards  Red Cards  Corner Kicks  Free Kicks  Offsides
Away Team      2     10     16             5          1             7           7         1
Home Team      0     19      7             1          0             8          15         1


# Prompting test

In [12]:
idx = 2

In [13]:
df = pd.read_json('../data/livesum/test.json')
%clear
print(textwrap.fill(df['text'][idx], width=100))

The game is underway with the start of the first half. Player27(Away Team) fouls Player2(Home Team),
winning a free kick in their own defensive half. Player10(Home Team) committed a foul. Player26(Away
Team) earns a free kick in the opponent's half. Away Team earns a corner kick. Player24(Away Team)'s
shot from the left side of the six yard box is saved in the center of the goal. The Away Team wins a
corner kick. The Away Team wins a corner kick. Player number 20 is currently being delayed in the
match due to an injury. The delay is finished and they are prepared to resume play. Player27(Away
Team) of the Away Team is caught offside after Player22(Away Team) attempts a through ball.
Player30(Away Team) is being delayed in the match due to an injury. Player9(Home Team) is currently
being delayed in the match due to an injury. The delay is finished and they are prepared to resume
play. Player7(Home Team)'s left footed shot from the left side of the box is blocked after a cross
from Playe

In [14]:
text = df['text'][idx]
atomic_out = ask_chatgpt(text=text,prompt_path="prompts/livesum_atomic.txt")
print(atomic_out)

The game begins with the start of the first half.  
Player27 (Away Team) fouls Player2 (Home Team), resulting in a free kick for the Away Team in their own defensive half.  
Player10 (Home Team) commits a foul.  
Player26 (Away Team) earns a free kick in the opponent's half.  
The Away Team earns a corner kick.  
Player24 (Away Team) takes a shot from the left side of the six-yard box, which is saved in the center of the goal.  
The Away Team wins a corner kick.  
The Away Team wins another corner kick.  
Player20 (Away Team) is delayed in the match due to an injury.  
The delay caused by Player20's injury ends, and play is prepared to resume.  
Player27 (Away Team) is caught offside after Player22 (Away Team) attempts a through ball.  
Player30 (Away Team) is delayed in the match due to an injury.  
Player9 (Home Team) is delayed in the match due to an injury.  
The delay caused by Player9's injury ends, and play is prepared to resume.  
Player7 (Home Team) takes a left-footed shot fr

In [15]:
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'w') as f:
    f.write(atomic_out)

In [16]:
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'r') as f:
    atomic_text = f.read()
header_out = ask_chatgpt(text=atomic_text,prompt_path="prompts/livesum_header.txt")
print(header_out)

{
  "row_headers": [
    "Player",
    "Team",
    "Event",
    "Foul",
    "Free Kick",
    "Shot",
    "Corner Kick",
    "Injury",
    "Yellow Card",
    "Score"
  ],
  "column_headers": [
    "Player Name",
    "Team Name",
    "Event Type",
    "Foul Committed",
    "Free Kick Awarded",
    "Shot Attempted",
    "Corner Kick Earned",
    "Injury Status",
    "Yellow Card Received",
    "Final Score"
  ]
}


In [17]:
with open('./model_outputs/gpt_livesum_test/header_each.txt', 'w') as f:
    f.write(header_out)

In [18]:
with open('./model_outputs/gpt_livesum_test/header_each.txt', 'r') as f:
    header_text = f.read()
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'r') as f:
    atomic_text = f.read()
input_text = header_text + '\n' + atomic_text
output_table = ask_chatgpt(text=input_text,prompt_path="prompts/livesum_table.txt")
print(output_table)

 | Player Name | Team Name | Event Type | Foul Committed | Free Kick Awarded | Shot Attempted | Corner Kick Earned | Injury Status | Yellow Card Received | Final Score |
 | Player27 | Away Team | Foul | Player2 | Yes | Not found | Not found | Not found | Not found | Not found |
 | Player10 | Home Team | Foul | Not found | Yes | Not found | Not found | Not found | Not found | Not found |
 | Player26 | Away Team | Free Kick | Not found | Yes | Not found | Not found | Not found | Not found | Not found |
 | Not found | Away Team | Corner Kick | Not found | Not found | Not found | Yes | Not found | Not found | Not found |
 | Player24 | Away Team | Shot | Not found | Not found | Yes | Not found | Not found | Not found | Not found |
 | Not found | Away Team | Corner Kick | Not found | Not found | Not found | Yes | Not found | Not found | Not found |
 | Not found | Away Team | Corner Kick | Not found | Not found | Not found | Yes | Not found | Not found | Not found |
 | Player20 | Away Team | 

In [19]:
convert_to_df(output_table)

,Player Name,Team Name,Event Type,Foul Committed,Free Kick Awarded,Shot Attempted,Corner Kick Earned,Injury Status,Yellow Card Received,Final Score
0,Player27,Away Team,Foul,Player2,Yes,Not found,Not found,Not found,Not found,Not found
1,Player10,Home Team,Foul,Not found,Yes,Not found,Not found,Not found,Not found,Not found
2,Player26,Away Team,Free Kick,Not found,Yes,Not found,Not found,Not found,Not found,Not found
3,Not found,Away Team,Corner Kick,Not found,Not found,Not found,Yes,Not found,Not found,Not found
4,Player24,Away Team,Shot,Not found,Not found,Yes,Not found,Not found,Not found,Not found
...,...,...,...,...,...,...,...,...,...,...
88,Not found,Home Team,Shot,Not found,Not found,Yes,Not found,Not found,Not found,Not found
89,Player13,Home Team,Shot,Not found,Not found,Yes,Not found,Not found,Not found,Not found
90,Player26,Away Team,Goal,Not found,Not found,Not found,Not found,Not found,Not found,2-0
91,Not found,Home Team,Score,Not found,Not found,Not found,Not found,Not found,Not found,0-2


In [20]:
print(df_table.to_string(index=False))

     Team  Goals  Shots  Fouls  Yellow Cards  Red Cards  Corner Kicks  Free Kicks  Offsides
Away Team      2     10     16             5          1             7           7         1
Home Team      0     19      7             1          0             8          15         1
